<a href="https://colab.research.google.com/github/Safwanbzs/image_dehazing-ml_noob/blob/main/3d_mesh_research_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# State-of-the-Art Literature Survey, Technical Guidelines, and Research Proposals for Resource-Efficient 3D Mesh Denoising

3D mesh denoising is a fundamental task in digital geometry processing, computer graphics, and computer vision. Triangular meshes acquired through 3D laser scanners, photogrammetry, depth sensors, or LiDAR inevitably suffer from measurement noise and structural corruption. Removing noise while preserving intrinsic high-frequency surface features, such as sharp edges, corners, and fine textures, remains a classic ill-posed problem.

While deep learning has achieved remarkable success in 3D geometry processing, existing state-of-the-art models rely heavily on over-parameterized Graph Neural Networks (GNNs), 3D volumetric convolutions, and iterative diffusion processes. These architectures require substantial GPU memory (VRAM) and high-end computational hardware, rendering them impractical for deployment on resource-restricted platforms such as low-end GPUs, embedded edge devices, or standard CPU setups.

Converting face normal fields into structured signals and filtering them via lightweight neural architectures offers a promising direction. This report presents a comprehensive literature survey of the domain, establishes the mathematical foundations of normal-to-signal conversion, provides practical technical guidelines for resource-constrained execution, and formulates three actionable research proposals complete with manuscript blueprints.

---

## State-of-the-Art Literature Survey in 3D Mesh Denoising

Research in 3D mesh denoising has evolved across three major technological paradigms: classical geometric optimization, data-driven Graph Neural Networks, and generative high-capacity architectures.

### Traditional Geometry-Based Filtering Methods

Classical mesh filtering algorithms operate directly on the discrete differential geometry of the mesh without offline dataset training. They rely on hand-crafted geometric priors, such as piecewise smoothness, bounded total variation, or local planarity.

Early approaches directly adjusted vertex positions via Laplacian smoothing, but this led to volume shrinkage and over-smoothing of sharp edges. To resolve this, two-stage paradigms emerged: filtering the face normal field first, followed by reconstructing vertex positions through normal integration.

1. **Bilateral and Guided Normal Filtering**:
Methods like Bilateral Mesh Denoising, [Bi-Normal Filtering](https://www.researchgate.net/publication/273394182_Bi-Normal_Filtering_for_Mesh_Denoising), and Guided Mesh Normal Filtering (GMNF) introduce spatial and range weights to filter face normals based on orientation similarity. They preserve sharp features by ensuring faces across distinct geometric boundaries do not average their normals together.
2. **Optimization and Variational Approaches**:
Variational mesh denoising frameworks formulate objective functions with sparse regularizers, such as $L_0$ or $L_1$ norms of normal differences, or Total Generalized Variation (TGV). These techniques enforce piecewise constant normal fields, which effectively preserve sharp crease lines on Computer-Aided Design (CAD) shapes, but often create artificial flat facets on organic or smooth free-form surfaces.

While traditional techniques require zero training memory and execute with minimal VRAM, they depend heavily on manual hyperparameter tuning for specific noise distributions and fail under severe non-Gaussian scanner artifacts.

### Graph Convolutional Neural Networks

The emergence of Geometric Deep Learning enabled neural networks to operate directly on irregular mesh graphs without converting them into structured 2D images or 3D grids.

1. **Facet-Graph Neural Networks**:
Architectures such as [DNF-Net](https://www.researchgate.net/publication/342548255_DNF-Net_a_Deep_Normal_Filtering_Network_for_Mesh_Denoising) process local patches of face normals. DNF-Net treats face normals in a local neighborhood as input and regresses noise-free face normals using multi-layer perceptrons (MLPs) paired with spatial spatial-descriptor encodings.
2. **Multi-Scale Graph Embedding Networks**:
Models like [ResGEM](https://www.researchgate.net/publication/379062020_ResGEM_Multi-scale_Graph_Embedding_Network_for_Residual_Mesh_Denoising) utilize multi-scale graph convolutions to learn residual face normal updates. By aggregating node features across multi-hop face adjacencies, ResGEM extracts both local feature details and global surface consistency.
3. **Volumetric Facet Networks**:
[NormalNet](https://arxiv.org/abs/1903.04015) maps local face normal neighborhoods into volumetric voxel grids, allowing 3D Convolutional Neural Networks (CNNs) to learn spatial filtering rules. However, 3D voxelization creates significant memory overhead and spatial quantization loss.

Graph-based methods demonstrate high noise robustness and generalization. However, computing dynamically updated graph adjacencies or maintaining deep message-passing layers for large meshes ($>100,000$ faces) causes GPU memory consumption to scale quadratically with graph depth.

### High-Capacity Mesh Transformers and Diffusion Architectures

Recent developments adapt generative paradigms and global attention mechanisms to geometry restoration.

1. **Mesh Denoising Transformers**:
The [Mesh Denoising Transformer](https://arxiv.org/html/2405.06536v1) replaces local neighborhood convolutions with self-attention mechanisms. By modeling long-range dependencies across distant mesh facets, transformers effectively disambiguate structural noise from geometric features.
2. **Structure-Preserving Diffusion Models**:
[DMESH](https://www.researchgate.net/publication/378523912_DMESH_A_Structure-Preserving_Diffusion_Model_for_3-D_Mesh_Denoising) presents a diffusion-based mesh denoiser that progressively removes noise from corrupted surfaces. Rather than diffusing vertices into isotropic Gaussian noise, DMESH constrains the diffusion path to preserve global geometry and uses 2D viewpoint projections to bypass irregular topology.

Although Transformers and Diffusion models achieve high visual fidelity, their computational complexity and VRAM requirements (>4 to 8 GB VRAM, long inference latency) restrict their utility in resource-constrained environments.

### Comparative Performance and Hardware Resource Taxonomy

The following table provides a direct comparison of current 3D mesh denoising paradigms across architectural design, feature preservation, memory consumption, and computational speed.

| Method Family | Representative Models | Input Domain | Feature Preservation Capability | Average VRAM Footprint | Processing Speed (100k Faces) |
| --- | --- | --- | --- | --- | --- |
| **Traditional Geometry** | GMNF, [Bi-Normal Filter](https://www.researchgate.net/publication/273394182_Bi-Normal_Filtering_for_Mesh_Denoising) | Face Normals | Moderate (Requires hyperparameter tuning) | $< 100$ MB | $< 2.0$ seconds (CPU) |
| **Volumetric CNNs** | [NormalNet](https://arxiv.org/abs/1903.04015) | Voxelized Facet Local Patch | Moderate to High | $\sim 2.5$ GB | $\sim 8.5$ seconds (GPU) |
| **Graph Neural Networks** | [DNF-Net](https://www.researchgate.net/publication/342548255_DNF-Net_a_Deep_Normal_Filtering_Network_for_Mesh_Denoising), [ResGEM](https://www.researchgate.net/publication/379062020_ResGEM_Multi-scale_Graph_Embedding_Network_for_Residual_Mesh_Denoising) | Facet Adjacency Graph | High | $\sim 1.8$ GB – $3.2$ GB | $\sim 3.0$ seconds (GPU) |
| **Mesh Transformers** | [Mesh Denoising Transformer](https://arxiv.org/html/2405.06536v1) | Global/Patch Facet Tokens | High | $> 4.0$ GB | $> 12.0$ seconds (GPU) |
| **Mesh Diffusion** | [DMESH](https://www.researchgate.net/publication/378523912_DMESH_A_Structure-Preserving_Diffusion_Model_for_3-D_Mesh_Denoising) | Multi-view 2D Projections | High | $> 6.0$ GB | $> 45.0$ seconds (GPU) |
| **Target Lightweight Signal** | *Proposed 1D-POSNet / L-GWRD* | Ordered 1D Normal Sequences | High (Feature-aware losses) | $< 350$ MB | $< 0.8$ seconds (CPU/GPU) |

---

## Theoretical Foundations of Surface Normal Signal Transformation

Understanding mesh normal denoising requires framing surface normals as vector-valued signals defined over discrete 2D manifold surfaces embedded in $\mathbb{R}^3$.

### Mathematical Representation of Mesh Normal Vectors

Let $\mathcal{M} = (\mathcal{V}, \mathcal{F}, \mathcal{E})$ represent a triangular surface mesh, where $\mathcal{V} = \{\mathbf{v}_i \in \mathbb{R}^3\}_{i=1}^V$ is the set of vertices, $\mathcal{F} = \{f_j\}_{j=1}^F$ is the set of triangular faces, and $\mathcal{E}$ is the set of edges.

Each triangular face $f_j$ defined by vertices $(\mathbf{v}_{j1}, \mathbf{v}_{j2}, \mathbf{v}_{j3})$ has an unnormalized face normal computed via the cross product:

$$\mathbf{u}_j = (\mathbf{v}_{j2} - \mathbf{v}_{j1}) \times (\mathbf{v}_{j3} - \mathbf{v}_{j1})$$

The unit face normal vector $\mathbf{n}_j \in \mathbb{S}^2$ is:

$$\mathbf{n}_j = \frac{\mathbf{u}_j}{\Vert{}\mathbf{u}_j\Vert{}_2}$$

The collection of face normals $\mathbf{N} = \{\mathbf{n}_1, \mathbf{n}_2, \dots, \mathbf{n}_F\}$ forms a discrete vector field over the mesh faces. Noise in vertex positions corrupts the face normal field. Because face normals represent first-order differential quantities of the surface, filtering the normal field $\mathbf{N}$ is significantly more stable and feature-preserving than directly smoothing raw 3D vertex coordinates $\mathcal{V}$.

### Signal Discretization and Local Neighborhood Ordering

To apply standard 1D or 2D convolutional filters to an irregular mesh, the local neighborhood around each face must be mapped into a structured tensor.

For a target face $f_i$, let $\mathcal{N}(f_i, k)$ denote its $k$-ring face neighborhood, defined as the set of faces reachable within $k$ topological steps across shared edges. Because the cardinality and ordering of $\mathcal{N}(f_i, k)$ vary across irregular meshes, a deterministic sorting function $\mathcal{S}$ is applied to order the neighbors into a fixed-length 1D sequence signal $\mathbf{S}_i \in \mathbb{R}^{K \times 3}$:

$$\mathbf{S}_i = \mathcal{S}\left( \{\mathbf{n}_j \mid f_j \in \mathcal{N}(f_i, k)\} \right) = [\mathbf{n}_{i,1}, \mathbf{n}_{i,2}, \dots, \mathbf{n}_{i,K}]^T$$

The sorting function $\mathcal{S}$ can be constructed using geometric or topological criteria:

* **Geodesic Distance & Polar Angle**: Sort neighboring faces by ascending Euclidean/geodesic distance from the centroid of $f_i$, breaking ties by polar angle relative to a projection plane defined by $\mathbf{n}_i$.
* **Spiral Traversal**: Traversing adjacent faces in a deterministic clockwise or counter-clockwise spiral pattern starting from the neighbor with the lowest index.

This formulation converts irregular 3D mesh normal structures into standard 1D multi-channel sequences, making them compatible with lightweight 1D convolutional layers, depthwise-separable 1D networks, or 1D State Space Models.

### Vertex Position Reconstruction and Error Minimization

Once the neural network predicts a clean set of face normals $\mathbf{\hat{N}} = \{\mathbf{\hat{n}}_1, \mathbf{\hat{n}}_2, \dots, \mathbf{\hat{n}}_F\}$, the vertex positions $\mathcal{V}$ must be updated to align with $\mathbf{\hat{N}}$.

Following [Sun et al.'s formulation](https://langbein.org/wp-content/uploads/2009/06/sun2007.pdf), the alignment error for an edge connecting vertex $\mathbf{v}_a$ and $\mathbf{v}_b$ belonging to face $f_j$ is defined by the orthogonality condition:

$$\mathbf{\hat{n}}_j \cdot (\mathbf{v}_a - \mathbf{v}_b) = 0$$

Updating vertex positions requires minimizing the global quadratic energy function across all faces:

$$E(\mathcal{V}) = \sum_{f_j \in \mathcal{F}} \sum_{(\mathbf{v}_a, \mathbf{v}_b) \in \partial f_j} \left( \mathbf{\hat{n}}_j \cdot (\mathbf{v}_a - \mathbf{v}_b) \right)^2$$

Differentiating $E(\mathcal{V})$ with respect to vertex position $\mathbf{v}_i$ yields the iterative gradient descent update rule:

$$\mathbf{v}_i^{(t+1)} = \mathbf{v}_i^{(t)} + \frac{1}{\vert{}\mathcal{F}(\mathbf{v}_i)\vert{}} \sum_{f_j \in \mathcal{F}(\mathbf{v}_i)} \mathbf{\hat{n}}_j \left[ \mathbf{\hat{n}}_j \cdot \left( \mathbf{c}_j^{(t)} - \mathbf{v}_i^{(t)} \right) \right]$$

where $\mathcal{F}(\mathbf{v}_i)$ is the set of faces sharing vertex $\mathbf{v}_i$, and $\mathbf{c}_j^{(t)} = \frac{1}{3} (\mathbf{v}_{j1}^{(t)} + \mathbf{v}_{j2}^{(t)} + \mathbf{v}_{j3}^{(t)})$ is the centroid of face $f_j$ at iteration $t$. This update rule converges rapidly (typically in 10 to 20 iterations) and requires minimal CPU/GPU memory overhead.

---

## Technical Implementation Guidelines for Resource-Constrained Environments

Executing deep learning models in memory-restricted environments (e.g., $< 2$ GB VRAM or integrated graphics) requires optimizing data structures, model architectures, and loss formulations.

### Memory-Efficient Data Structures and Patching Strategies

1. **Avoid Full Mesh Adjacency Graphs**:
Do not build or load full $F \times F$ global sparse adjacency matrices into GPU memory during training or inference. Instead, stream local 1D normal patches in small batches.
2. **On-the-Fly Patch Extraction**:
Precompute the top-$K$ neighboring indices for each face on the CPU during data pre-processing and store them as compact 16-bit integer arrays (`uint16`). During inference, load face normal vectors dynamically using fast index gather operations:

In [ ]:
# Low-memory patch gathering in PyTorch
# face_normals: [F, 3], neighbor_indices: [F, K]
patch_signals = face_normals[neighbor_indices] # Output shape: [F, K, 3]

3. **Inference Chunking**:
Process large meshes in streaming chunks (e.g., 10,000 faces per chunk). This bounds peak VRAM allocation to $< 200$ MB regardless of total mesh size.

### Lightweight Network Architectures and Model Compression

1. **Depthwise Separable 1D Convolutions**:
Replace standard 1D convolutions with depthwise separable 1D convolutions. Depthwise separable blocks reduce parameter count and computational complexity by $80-90\%$ without sacrificing expressive capacity.
2. **Inverted Bottleneck Residual Blocks**:
Adopt MobileNetV3-style inverted residual blocks, expanding channels internally with $1 \times 1$ convolutions, applying $3 \times 1$ depthwise filtering, and projecting back to lower channel dimensions.
3. **Quantization and Half-Precision (FP16/INT8)**:
Train or export models in half-precision (`float16`) or quantized `INT8` ONNX formats. This halves memory bandwidth and speeds up execution on modern low-power hardware.

### Feature-Preserving Loss Functions in Signal Space

Standard mean squared error ($L_2$) loss on face normals causes over-smoothing near sharp geometry. To preserve features without adding network parameter complexity, use a multi-term loss function combining angle loss, gradient loss, and structural loss:

1. **Cosine Similarity Angle Loss**:

$$\mathcal{L}_{\text{angle}} = \frac{1}{F} \sum_{j=1}^F \left( 1 - \mathbf{n}_j^{\text{gt}} \cdot \mathbf{\hat{n}}_j \right)$$

2. **1D Signal Gradient Loss**:
To preserve sharp transitions across normal boundaries, enforce consistency on the first derivative of the 1D ordered normal sequence:

$$\mathcal{L}_{\text{grad}} = \frac{1}{F \cdot (K-1)} \sum_{j=1}^F \sum_{k=1}^{K-1} \left\Vert{} (\mathbf{\hat{S}}_{j, k+1} - \mathbf{\hat{S}}_{j, k}) - (\mathbf{S}_{j, k+1}^{\text{gt}} - \mathbf{S}_{j, k}^{\text{gt}}) \right\Vert{}_1$$

3. **Total Loss Formulation**:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{angle}} + \lambda_{\text{grad}} \mathcal{L}_{\text{grad}}$$

where $\lambda_{\text{grad}} = 0.2$ balances global normal alignment with sharp boundary preservation.

---

## Concrete Research Proposals for Lightweight Mesh Denoising

The following three research proposals outline novel, resource-efficient methodologies for 3D mesh denoising. Each proposal includes a formal problem statement, architectural blueprint, evaluation protocol, and expected contribution.

---

### Proposal 1: 1D Patch-Ordered Signal Convolutional Network (1D-POSNet)

#### Problem Statement

Current state-of-the-art mesh denoisers utilize heavy 3D volumetric convolutions ([NormalNet](https://arxiv.org/abs/1903.04015)) or multi-layer graph convolutions ([ResGEM](https://www.researchgate.net/publication/379062020_ResGEM_Multi-scale_Graph_Embedding_Network_for_Residual_Mesh_Denoising)), resulting in high GPU memory usage. There is a need for a lightweight model that reformulates 3D facet neighborhoods into structured 1D signals to enable high-speed normal filtering under low VRAM constraints.

#### Proposed Architectural Blueprint

1. **Polar-Geodesic 1D Signal Extraction**:
For each face $f_i$, gather its $K=16$ immediate topological neighbors. Project neighbor centroids onto the tangent plane defined by $\mathbf{n}_i$. Order the neighbors sequentially based on polar angle $\theta \in [0, 2\pi)$ relative to an arbitrary reference axis. This produces a $16 \times 3$ matrix representing a 1D sequence signal.
2. **Ultra-Lightweight 1D-CNN Architecture**:
Pass the $16 \times 3$ signal through a 4-layer 1D CNN with depthwise-separable convolutions, inverted residual bottlenecks, and squeeze-and-excitation (SE) attention blocks.
* **Input**: `[Batch, 3, 16]`
* **Layer 1**: Conv1D(3 $\to$ 32, kernel=3, stride=1, padding=1) + BatchNorm + ReLU
* **Layer 2**: DepthwiseSeparableConv1D(32 $\to$ 64, kernel=3) + SE-Block
* **Layer 3**: DepthwiseSeparableConv1D(64 $\to$ 32, kernel=3)
* **Layer 4**: Conv1D(32 $\to$ 3, kernel=1) + L2-Normalization
* **Total Parameters**: $\sim 85,000$ ($< 0.35$ MB model size).


3. **Vertex Reconstruction**:
Apply Sun et al.'s iterative vertex update rule for 10 iterations on CPU/GPU.

#### Benchmark Protocol and Evaluation Plan

* **Datasets**:
* Synthetic: ModelNet40 test split (CAD models) corrupted with $0.5\%, 1.0\%, 1.5\%$ Gaussian noise.
* Real-world: Scanned models from the [Thingi10K dataset](https://arxiv.org/abs/1605.04797).


* **Baselines**: GMNF, [Bi-Normal Filter](https://www.researchgate.net/publication/273394182_Bi-Normal_Filtering_for_Mesh_Denoising), [DNF-Net](https://www.researchgate.net/publication/342548255_DNF-Net_a_Deep_Normal_Filtering_Network_for_Mesh_Denoising).
* **Metrics**: Mean Normal Error (MNE), Hausdorff Distance (HD), Chamfer Distance (CD), Peak VRAM (MB), and Execution Time (ms).

#### Expected Contributions

* Demonstration that 1D patch-ordered sequence filtering achieves competitive denoising accuracy with $95\%$ fewer parameters than 3D/Graph networks.
* An open-source, ultra-fast Python/C++ pipeline capable of denoising 100k-face meshes in under 0.5 seconds on integrated GPUs.

---

### Proposal 2: Self-Supervised Single-Mesh Normal Filtering (S3M-Filter)

#### Problem Statement

Supervised learning models require paired datasets of noisy meshes and ground-truth clean meshes. In real-world 3D scanning, ground-truth clean geometry is rarely available. Existing self-supervised methods on graphs are computationally expensive and struggle with overfitting on single meshes.

#### Proposed Architectural Blueprint

1. **Noise2Noise Adaptation for Normal Fields**:
Given a single noisy mesh $\mathcal{M}_{\text{noisy}}$, generate two distinct noisy instances $\mathcal{N}_1$ and $\mathcal{N}_2$ by adding independent, zero-mean synthetic perturbations to the input normal field.
2. **Tiny Implicit MLP Denoiser**:
Construct a tiny Coordinate-MLP (3 hidden layers, 64 channels each, with sine activation functions or Fourier features). The network takes local face centroids $\mathbf{c}_i \in \mathbb{R}^3$ as input and predicts clean normal vectors $\mathbf{\hat{n}}_i$.
* **Loss Function**: Train the network on the single mesh by minimizing the $L_1$ discrepancy between predictions and the two noisy targets:



$$\mathcal{L}_{\text{self}} = \Vert{}\mathbf{\hat{N}}(\mathbf{C}) - \mathcal{N}_1\Vert{}_1 + \Vert{}\mathbf{\hat{N}}(\mathbf{C}) - \mathcal{N}_2\Vert{}_1 + \gamma \Vert{}\nabla \mathbf{\hat{N}}(\mathbf{C})\Vert{}_1$$

3. **Zero-Shot On-the-Fly Optimization**:
The network is trained directly on the input noisy mesh for $200-300$ iterations (taking $< 3$ seconds on low-end hardware) and discarded after inference, eliminating the need for pre-saved model weights or offline datasets.

#### Benchmark Protocol and Evaluation Plan

* **Datasets**: Real Kinect and LiDAR scanned point-cloud/mesh models (e.g., Paris-rue-Madame dataset).
* **Baselines**: Unsupervised Bilateral Normal Filter, Total Variation (TV) mesh filter.
* **Metrics**: MNE, Edge Sharpness Index (ESI), Memory usage, Convergence speed (iterations vs. loss).

#### Expected Contributions

* A zero-shot, single-mesh denoising framework that operates without ground-truth training data.
* A low-memory footprint ($< 150$ MB RAM) suitable for on-device deployment in handheld 3D scanners.

---

### Proposal 3: Lightweight Graph-Wavelet Residual Denoising (L-GWRD)

#### Problem Statement

Graph Neural Networks often act as low-pass filters, removing high-frequency noise but unintentionally smoothing out sharp structural edges. Spectral graph wavelets can isolate high-frequency features, but full spectral decomposition ($O(V^3)$) is computationally prohibitive for large meshes.

#### Proposed Architectural Blueprint

1. **Chebyshev Polynomial Graph-Wavelet Approximation**:
Approximate spectral graph wavelets over local face-adjacency graph patches using low-degree Chebyshev polynomials ($K=3$). This avoids eigenvalue decomposition, reducing computational complexity to $O(K \cdot \vert{}\mathcal{E}\vert{})$.
2. **Dual-Stream High/Low-Frequency Filter**:
Construct a lightweight two-stream architecture:
* **Low-Frequency Stream**: 2-layer GCN capturing smooth global topology.
* **High-Frequency Stream**: 3-order Chebyshev Graph Wavelet block capturing sharp geometric transitions and filtering high-frequency noise components.


3. **Residual Fusion Block**:
Combine low- and high-frequency stream outputs via a lightweight channel-attention gate to predict residual normal corrections $\Delta \mathbf{n}_i$:

$$\mathbf{\hat{n}}_i = \frac{\mathbf{n}_i^{\text{noisy}} + \Delta \mathbf{n}_i}{\Vert{}\mathbf{n}_i^{\text{noisy}} + \Delta \mathbf{n}_i\Vert{}_2}$$

#### Benchmark Protocol and Evaluation Plan

* **Datasets**: CAD meshes with sharp edges (Fandisk, Block, Joint) mixed with smooth organic models (Stanford Bunny, Armadillo).
* **Baselines**: [ResGEM](https://www.researchgate.net/publication/379062020_ResGEM_Multi-scale_Graph_Embedding_Network_for_Residual_Mesh_Denoising), [DNF-Net](https://www.researchgate.net/publication/342548255_DNF-Net_a_Deep_Normal_Filtering_Network_for_Mesh_Denoising), TGV Mesh Filter.
* **Metrics**: Corner Preservation Error (CPE), MNE, Chamfer Distance, VRAM allocation during graph forward passes.

#### Expected Contributions

* A mathematically grounded spectral-wavelet filtering approach optimized for irregular meshes.
* An edge-preserving mesh denoiser that avoids over-smoothing while maintaining low VRAM consumption ($< 400$ MB).

---

## Structural Outline for Master Thesis and Publication

Below is a proposed structural blueprint for a academic thesis or peer-reviewed journal manuscript based on the proposed lightweight research directions.

### Proposed Chapter Structure and Manuscript Blueprint

```text
TITLE: Resource-Efficient 3D Mesh Denoising via Lightweight Face Normal Signal Transformation

ABSTRACT
  - Background & Motivation (Ill-posed geometry denoising, high computational cost of modern GNNs)
  - Proposed Methodology (1D Signal transformation, lightweight networks, feature-preserving losses)
  - Key Findings & Results (MNE accuracy, 90% reduction in VRAM, real-time inference speed)

CHAPTER 1: INTRODUCTION
  1.1 Overview of 3D Geometry Processing & Scanning Artifacts
  1.2 Statement of the Problem (Resource restrictions in real-world deployment)
  1.3 Research Objectives & Key Questions
  1.4 Summary of Contributions

CHAPTER 2: LITERATURE REVIEW
  2.1 Classical Surface Normal Filtering & Optimization (Bilateral, Guided, TGV)
  2.2 Learning-Based Geometry Denoising (3D CNNs, GNNs, Transformers, Diffusion)
  2.3 Review of Graph Signal Processing & Patch Normal Representations
  2.4 Identification of Literature Gaps in Resource-Constrained Environments

CHAPTER 3: THEORETICAL FORMULATION & SIGNAL TRANSFORMATION
  3.1 Mathematical Principles of Face Normal Fields & Differential Geometry
  3.2 Polar-Geodesic Local Neighborhood Ordering Algorithm
  3.3 Spectral & Spatial 1D Signal Conversion
  3.4 Normal-Driven Vertex Position Optimization Scheme

CHAPTER 4: LIGHTWEIGHT ARCHITECTURE & IMPLEMENTATION
  4.1 Architectural Blueprint (1D-POSNet / S3M-Filter / L-GWRD)
  4.2 Depthwise-Separable Convolutions & Memory-Efficient Data Pipelines
  4.3 Feature-Preserving Loss Functions (Gradient & Cosine Angle Losses)
  4.4 Hardware Optimization (Quantization, FP16 execution, CPU-GPU streaming)

CHAPTER 5: EXPERIMENTAL RESULTS & ANALYSIS
  5.1 Experimental Setup (Datasets: ModelNet40, Thingi10K, Real Scans; Hardware specifications)
  5.2 Quantitative Comparisons (MNE, Hausdorff Distance, Chamfer Distance)
  5.3 Memory and Computational Complexity Benchmarks (VRAM, Execution Time, Parameter Count)
  5.4 Qualitative Visual Analysis (CAD Sharp Creases vs. Organic Curved Surfaces)
  5.5 Ablation Studies (Impact of Patch Size K, Loss Components, Neighborhood Sorting Schemes)

CHAPTER 6: DISCUSSION & FUTURE WORK
  6.1 Analysis of Strengths and Limitations
  6.2 Practical Deployment Considerations on Embedded & Mobile Platforms
  6.3 Future Research Directions (Extension to Neural Radiance Fields / 3D Gaussian Splatting)

CHAPTER 7: CONCLUSION
  7.1 Summary of Findings

REFERENCES

```

---

## References

1. Pengbo Bo et al., [Mesh Denoising of Developable Surfaces with Curved Foldings](https://www.researchgate.net/publication/382398424_Mesh_denoising_of_developable_surfaces_with_curved_foldings), ResearchGate, 2024.
2. X. Zheng et al., [Multi-Scale Graph Embedding Network for Residual Mesh Denoising (ResGEM)](https://www.researchgate.net/publication/379062020_ResGEM_Multi-scale_Graph_Embedding_Network_for_Residual_Mesh_Denoising), ResearchGate, 2024.
3. K. Zhang et al., [A Structure-Preserving Diffusion Model for 3-D Mesh Denoising (DMESH)](https://www.researchgate.net/publication/378523912_DMESH_A_Structure-Preserving_Diffusion_Model_for_3-D_Mesh_Denoising), ResearchGate, 2024.
4. L. Wei et al., [Mesh Denoising Transformer](https://arxiv.org/html/2405.06536v1), arXiv Preprint, 2024.
5. H. Ben-Hamu et al., [DNF-Net: a Deep Normal Filtering Network for Mesh Denoising](https://www.researchgate.net/publication/342548255_DNF-Net_a_Deep_Normal_Filtering_Network_for_Mesh_Denoising), IEEE TVCG / ResearchGate, 2020.
6. Y. Wang et al., [NormalNet: Learning-based Normal Filtering for Mesh Denoising](https://arxiv.org/abs/1903.04015), arXiv Preprint, 2019.
7. W. Zhang et al., [Bi-Normal Filtering for Mesh Denoising](https://www.researchgate.net/publication/273394182_Bi-Normal_Filtering_for_Mesh_Denoising), ResearchGate, 2015.
8. X. Sun et al., [Fast and Effective Feature-Preserving Mesh Denoising](https://langbein.org/wp-content/uploads/2009/06/sun2007.pdf), IEEE TVCG / Langbein Lab, 2007.
9. Q. Zhou and D. Jacobson, [Thingi10K: A Dataset of 10,000 3D-Printing Models](https://arxiv.org/abs/1605.04797), arXiv / SGP, 2016.
10. J. Arvanitis et al., [3D Mesh Pre-Processing Method Based on Feature Point Extraction and Anisotropic Vertex Denoising](https://www.mdpi.com/2072-4292/13/11/2145), MDPI Remote Sensing, 2021.